# Diffusion models (DDPM) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## forward noising and learned denoising
Score models (10.11) provide the denoising direction, Gaussian algebra supplies the closed-form noisy sample, and neural nets predict noise or clean data. Classifier-free guidance (10.13) and latent diffusion (10.14) are direct extensions.

In [ ]:
# 1) forward noising combines signal and Gaussian noise
x0=2.; abar=.64; eps=-.5; xt=math.sqrt(abar)*x0+math.sqrt(1-abar)*eps
plt.figure(figsize=(4,3)); plt.bar(['signal','noise','x_t'],[math.sqrt(abar)*x0,math.sqrt(1-abar)*eps,xt]); plt.title('DDPM forward formula')
assert abs(xt-1.3)<1e-12

In [ ]:
# 2) signal decays along a schedule
abars=np.linspace(1,.01,80); signal=np.sqrt(abars)*x0
plt.figure(figsize=(5,3)); plt.plot(abars,signal); plt.xlabel('alpha_bar'); plt.ylabel('signal term'); plt.title('less alpha_bar means less clean signal')
assert signal[-1]<signal[0]

In [ ]:
# 3) noisy samples spread as t increases
rng=np.random.default_rng(0); eps=rng.normal(size=200); xts=math.sqrt(.25)*1.0+math.sqrt(.75)*eps
plt.figure(figsize=(5,3)); plt.hist(xts,bins=25); plt.title('forward process distribution')
assert np.std(xts)>0.7

In [ ]:
# 4) predicting epsilon lets us estimate x0
xt=1.3; eps_pred=-.5; x0_hat=(xt-math.sqrt(.36)*eps_pred)/math.sqrt(.64)
plt.figure(figsize=(4,3)); plt.bar(['true x0','estimated x0'],[2,x0_hat]); plt.title('recover clean value from predicted noise')
assert abs(x0_hat-2)<1e-12

In [ ]:
# 5) iterative denoising can approach data from noise
x=2.5; path=[x]
for a in [.2,.35,.5,.7,.9]: x=math.sqrt(a)*1.0+math.sqrt(1-a)*(x-1); path.append(x)
plt.figure(figsize=(5,3)); plt.plot(path,'o-'); plt.title('toy reverse steps')
assert abs(path[-1]-1)<abs(path[0]-1)